In [21]:
import requests
import pandas as pd

def get_books():
    url = "https://www.anapioficeandfire.com/api/books"
    response = requests.get(url)
    if response.status_code == 200:
        books_data = response.json()
        return pd.DataFrame(books_data)
    else:
        print(f"Ошибка при получении данных о книгах: {response.status_code}")
        return pd.DataFrame()

books_df = get_books()
print("Книги:")
print(books_df.head())

def get_houses():
    url = "https://www.anapioficeandfire.com/api/houses"
    response = requests.get(url)
    if response.status_code == 200:
        houses_data = response.json()
        return pd.DataFrame(houses_data)
    else:
        print(f"Ошибка при получении данных о домах: {response.status_code}")
        return pd.DataFrame()

houses_df = get_houses()
print("\nДома Вестероса:")
print(houses_df.head())

def get_houses_with_motto():
    url = "https://www.anapioficeandfire.com/api/houses?hasWords=true"
    response = requests.get(url)
    if response.status_code == 200:
        houses_data = response.json()
        return pd.DataFrame(houses_data)
    else:
        print(f"Ошибка при получении данных о домах с девизом: {response.status_code}")
        return pd.DataFrame()

houses_with_motto_df = get_houses_with_motto()
print("\nДома Вестероса с девизом:")
print(houses_with_motto_df.head())

Книги:
                                             url               name  \
0  https://www.anapioficeandfire.com/api/books/1  A Game of Thrones   
1  https://www.anapioficeandfire.com/api/books/2   A Clash of Kings   
2  https://www.anapioficeandfire.com/api/books/3  A Storm of Swords   
3  https://www.anapioficeandfire.com/api/books/4   The Hedge Knight   
4  https://www.anapioficeandfire.com/api/books/5  A Feast for Crows   

             isbn                authors  numberOfPages  \
0  978-0553103540  [George R. R. Martin]            694   
1  978-0553108033  [George R. R. Martin]            768   
2  978-0553106633  [George R. R. Martin]            992   
3  978-0976401100  [George R. R. Martin]            164   
4  978-0553801507  [George R. R. Martin]            784   

                   publisher        country     mediaType  \
0               Bantam Books  United States     Hardcover   
1               Bantam Books  United States      Hardback   
2               Bantam Books

In [22]:
!pip install psycopg2-binary

import psycopg2
import requests
import pandas as pd

# Параметры подключения к базе данных
host = 'hh-pgsql-public.ebi.ac.uk'
port = 5432
database = 'pfmegrnargs'
user = 'reader'
password = 'NWDMCE5xdipIjRrp'

# Подключение к базе данных
def connect_to_db():
    try:
        conn = psycopg2.connect(dbname=database, user=user, password=password, host=host)
        return conn
    except Exception as e:
        print(f"Ошибка подключения к базе данных: {e}")
        return None

 #1.  Выполняем SQL-запрос для получения 10 строк из таблицы rnc_database
def get_rnc_data(conn):
    try:
        with conn.cursor() as cursor:
            cursor.execute("SELECT * FROM rnc_database LIMIT 10")
            rows = cursor.fetchall()

            # Получаем имена столбцов
            cursor.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'rnc_database'")
            columns = [col[0] for col in cursor.fetchall()]

            return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return pd.DataFrame()

def get_specific_columns(conn):
    try:
        with conn.cursor() as cursor:
            cursor.execute("""
                SELECT display_name, num_sequences, num_organisms, url
                FROM rnc_database
                LIMIT 10
            """)
            rows = cursor.fetchall()
            return pd.DataFrame(rows, columns=['display_name', 'num_sequences', 'num_organisms', 'url'])
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return pd.DataFrame()

# 1. Подключаемся к БД
conn = connect_to_db()
if conn:
    # Получаем 10 строк из таблицы
    rnc_df = get_rnc_data(conn)
    print("\n Первые 10 строк из rnc_database:")
    print(rnc_df)

    # Получаем конкретные столбцы
    specific_columns_df = get_specific_columns(conn)
    print("\n Конкретные столбцы для 10 строк:")
    print(specific_columns_df)

    # Закрываем соединение
    conn.close()
else:
    print("Не удалось подключиться к базе данных")


Первые 10 строк из rnc_database:
   id  timestamp userstamp      descr  current_release full_descr alive  \
0  21 2017-05-02    RNACEN    NONCODE              146    NONCODE     Y   
1   5 2017-05-17    RNACEN       VEGA               98       VEGA     N   
2  26 2017-05-01    RNACEN    GENCODE              450    GENCODE     N   
3   1 2017-05-01    RNACEN        ENA              968        ENA     Y   
4  14 2017-05-01    RNACEN       TAIR              982       TAIR     Y   
5   9 2017-05-01    RNACEN     REFSEQ              969     RefSeq     Y   
6  41 2017-05-01    RNACEN  GENECARDS              978  MalaCards     Y   
7  10 2017-05-01    RNACEN        RDP               85        RDP     Y   
8  20 2017-05-01    RNACEN  LNCIPEDIA              935  LNCipedia     Y   
9  15 2017-05-02    RNACEN   WORMBASE              970   WormBase     Y   

  for_release display_name  project_id  avg_length  min_length  max_length  \
0                  NONCODE                  1130.0       201.0